# Util class for dynamic pricing agents

> Provides util functions for dynamic pricing agents

In [ ]:
#| default_exp agents.dynamic_pricing.utils

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import logging

from abc import ABC, abstractmethod
from typing import Union, Optional, List
import numpy as np
import joblib
import os
import torch 

In [ ]:
#| export
class GLMLink:
    """
    A class to represent a link function used in statistical models.
    Attributes
    ----------
    g : callable
        The link function g(x).
    g_inv : callable
        The inverse of the link function g⁻¹(x).
    g_prime : callable
        The derivative of the link function g'(x).
    link : str
        A string representing the type of link function.
    v : callable
        A function that computes the variance (derivative of the inverse link function).
    Methods
    -------
    __call__(x)
        Applies the link function g to the input x.
    """
    def __init__(self, g, g_inv, g_prime, link):
        self.g = g          # Link function g(x)
        self.g_inv = g_inv  # Inverse link function g⁻¹(x)
        self.g_prime = g_prime  # Derivative of the link function g'(x)
        self.link = link
        def v(x):
            return self.g_prime(self.g_inv(x))
        self.v = v
    def __call__(self, x):
        return self.g(x)

In [ ]:
#| export
def get_price_function(function_form="linear"):
    if function_form == "linear":
        def price_function(x, alpha, beta):
            assert len(alpha) == len(x) and len(beta) == len(x)
            return np.array(-np.divide(np.dot(alpha, x), 2*np.dot(beta, x)))
        return price_function
    if function_form == "log":
        def price_function(x, alpha, beta):
            assert len(alpha) == len(x) and len(beta) == len(x)
            return np.array(-np.divide(np.dot(alpha, x), np.dot(beta, x)))
        return price_function